# 08 — Ensemble: Blend, Stack & Champion Selection

Este notebook implementa **3 estratégias de ensemble** para combinar os modelos treinados anteriormente:

1. **Média Simples (Top-3)** — combina as predições dos melhores modelos com pesos iguais
2. **Blend (SLSQP Weight Optimization)** — otimiza os pesos de cada modelo via programação não-linear
3. **Stacking (LR Meta-Learner)** — treina uma Regressão Logística sobre as predições base como meta-features

Ao final, selecionamos o **Champion** com base no maior KS no conjunto OOT (out-of-time).

In [ ]:
import numpy as np
import pandas as pd
import pickle
import json
import os
import time
from scipy.optimize import minimize
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import sys

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
from config import *
from utils import *

In [ ]:
# ---------------------------------------------------------------------------
# Carregar os 5 modelos treinados
# ---------------------------------------------------------------------------
MODEL_DIR = os.path.join(ARTIFACT_DIR, 'models')

model_files = sorted([f for f in os.listdir(MODEL_DIR) if f.endswith('.pkl')])
print(f"Arquivos de modelo encontrados: {model_files}")

models = {}
for mf in model_files:
    name = mf.replace('.pkl', '')
    with open(os.path.join(MODEL_DIR, mf), 'rb') as fh:
        models[name] = pickle.load(fh)

model_names = list(models.keys())
print(f"Modelos carregados ({len(models)}): {model_names}")

## Predicoes Base

Geramos as predições (probabilidade da classe positiva) de cada modelo nos conjuntos **OOS** (out-of-sample, para otimização de pesos) e **OOT** (out-of-time, para avaliação final).

In [ ]:
# ---------------------------------------------------------------------------
# Carregar Feature Store e realizar split temporal
# ---------------------------------------------------------------------------
fs_path = LOCAL_DATA_PATH
df = pd.read_parquet(fs_path)
print(f"Feature Store carregada: {df.shape}")

# Split temporal
df_train = df[df['SAFRA'].isin(TRAIN_SAFRAS)].copy()
df_oos   = df[df['SAFRA'] == OOS_SAFRA].copy()
df_oot   = df[df['SAFRA'].isin(OOT_SAFRAS)].copy()

print(f"Train: {len(df_train):,} | OOS: {len(df_oos):,} | OOT: {len(df_oot):,}")

features = SELECTED_FEATURES
X_oos = df_oos[features]
y_oos = df_oos[TARGET]
X_oot = df_oot[features]
y_oot = df_oot[TARGET]

# Gerar predições base
preds_oos = pd.DataFrame(index=df_oos.index)
preds_oot = pd.DataFrame(index=df_oot.index)

for name, model in models.items():
    preds_oos[name] = model.predict_proba(X_oos)[:, 1]
    preds_oot[name] = model.predict_proba(X_oot)[:, 1]

print(f"\nPredições OOS shape: {preds_oos.shape}")
print(f"Predições OOT shape: {preds_oot.shape}")
print("\nCorrelação entre predições OOT:")
print(preds_oot.corr().round(3))

## Estrategia 1: Media Simples (Top-3)

A abordagem mais simples: calcular a **média aritmética** das probabilidades preditas por todos os modelos. Não requer otimização e serve como baseline para as demais estratégias.

In [ ]:
# ---------------------------------------------------------------------------
# Estratégia 1 — Média Simples
# ---------------------------------------------------------------------------
pred_avg_oot = preds_oot.mean(axis=1).values

ks_avg  = compute_ks(y_oot, pred_avg_oot)
auc_avg = roc_auc_score(y_oot, pred_avg_oot)
gini_avg = compute_gini(y_oot, pred_avg_oot)

print("=== Estratégia 1: Média Simples ===")
print(f"  KS  OOT: {ks_avg:.4f}")
print(f"  AUC OOT: {auc_avg:.4f}")
print(f"  Gini OOT: {gini_avg:.2f}")

## Estrategia 2: Blend (SLSQP Weight Optimization)

Otimizamos os pesos de cada modelo no conjunto **OOS** usando o método **SLSQP** (Sequential Least Squares Programming). A função objetivo é **maximizar o KS** (minimizamos o negativo). As restrições garantem que os pesos somem 1 e que cada peso fique entre 0.01 e 1.0.

In [ ]:
# ---------------------------------------------------------------------------
# Estratégia 2 — Blend (SLSQP)
# ---------------------------------------------------------------------------
n_models = len(model_names)

def neg_ks_objective(weights):
    """Função objetivo: negativo do KS (para minimização)."""
    blended = np.dot(preds_oos.values, weights)
    return -compute_ks(y_oos, blended)

constraints = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0}
bounds = [(0.01, 1.0)] * n_models
x0 = np.ones(n_models) / n_models

result = minimize(
    neg_ks_objective,
    x0,
    method='SLSQP',
    bounds=bounds,
    constraints=constraints,
    options={'maxiter': 500, 'ftol': 1e-9}
)

optimal_weights = result.x
print("Pesos ótimos (SLSQP):")
for name, w in zip(model_names, optimal_weights):
    print(f"  {name}: {w:.4f}")

# Avaliar no OOT
pred_blend_oot = np.dot(preds_oot.values, optimal_weights)

ks_blend  = compute_ks(y_oot, pred_blend_oot)
auc_blend = roc_auc_score(y_oot, pred_blend_oot)
gini_blend = compute_gini(y_oot, pred_blend_oot)

print(f"\n=== Estratégia 2: Blend (SLSQP) ===")
print(f"  KS  OOT: {ks_blend:.4f}")
print(f"  AUC OOT: {auc_blend:.4f}")
print(f"  Gini OOT: {gini_blend:.2f}")

## Estrategia 3: Stacking (LR Meta-Learner)

Treinamos uma **Regressão Logística** (C=1.0, penalidade L2) sobre as predições dos modelos base como **meta-features**. O meta-learner é ajustado no OOS e avaliado no OOT.

In [ ]:
# ---------------------------------------------------------------------------
# Estratégia 3 — Stacking (Meta-Learner)
# ---------------------------------------------------------------------------
meta_learner = LogisticRegression(C=1.0, penalty='l2', solver='lbfgs', max_iter=1000)
meta_learner.fit(preds_oos.values, y_oos)

pred_stack_oot = meta_learner.predict_proba(preds_oot.values)[:, 1]

ks_stack  = compute_ks(y_oot, pred_stack_oot)
auc_stack = roc_auc_score(y_oot, pred_stack_oot)
gini_stack = compute_gini(y_oot, pred_stack_oot)

print("=== Estratégia 3: Stacking (LR Meta-Learner) ===")
print(f"  KS  OOT: {ks_stack:.4f}")
print(f"  AUC OOT: {auc_stack:.4f}")
print(f"  Gini OOT: {gini_stack:.2f}")
print(f"\nCoeficientes do meta-learner:")
for name, coef in zip(model_names, meta_learner.coef_[0]):
    print(f"  {name}: {coef:.4f}")

## Selecao do Champion

Comparamos as 3 estratégias e selecionamos o **Champion** com base no **maior KS no OOT**.

In [ ]:
# ---------------------------------------------------------------------------
# Comparação e seleção do Champion
# ---------------------------------------------------------------------------
strategies = {
    'simple_avg': {'ks': ks_avg, 'auc': auc_avg, 'gini': gini_avg, 'preds': pred_avg_oot},
    'blend':      {'ks': ks_blend, 'auc': auc_blend, 'gini': gini_blend, 'preds': pred_blend_oot},
    'stack':      {'ks': ks_stack, 'auc': auc_stack, 'gini': gini_stack, 'preds': pred_stack_oot},
}

comparison_df = pd.DataFrame({
    'Estrategia': list(strategies.keys()),
    'KS_OOT':  [s['ks'] for s in strategies.values()],
    'AUC_OOT': [s['auc'] for s in strategies.values()],
    'Gini_OOT': [s['gini'] for s in strategies.values()],
}).sort_values('KS_OOT', ascending=False)

print("=== Comparação das Estratégias ===")
print(comparison_df.to_string(index=False))

champion_name = comparison_df.iloc[0]['Estrategia']
print(f"\n>>> Champion selecionado: {champion_name} (KS OOT = {comparison_df.iloc[0]['KS_OOT']:.4f})")

In [ ]:
# ---------------------------------------------------------------------------
# Classe _EnsembleModel
# ---------------------------------------------------------------------------
class _EnsembleModel:
    """Modelo ensemble com suporte a blend e stack."""

    def __init__(self, base_models: dict, mode: str = 'blend',
                 weights: np.ndarray = None, meta_learner=None):
        self.base_models = base_models
        self.mode = mode
        self.weights = weights
        self.meta_learner = meta_learner
        self.model_names = list(base_models.keys())

    def predict_proba(self, X):
        """Gera probabilidades combinadas."""
        base_preds = np.column_stack([
            m.predict_proba(X)[:, 1] for m in self.base_models.values()
        ])

        if self.mode == 'simple_avg':
            proba_1 = base_preds.mean(axis=1)
        elif self.mode == 'blend':
            proba_1 = np.dot(base_preds, self.weights)
        elif self.mode == 'stack':
            proba_1 = self.meta_learner.predict_proba(base_preds)[:, 1]
        else:
            raise ValueError(f"Modo desconhecido: {self.mode}")

        proba_0 = 1 - proba_1
        return np.column_stack([proba_0, proba_1])

    def save(self, path: str):
        """Salva o ensemble em disco."""
        with open(path, 'wb') as fh:
            pickle.dump(self, fh)
        print(f"Ensemble salvo em: {path}")

    @staticmethod
    def load(path: str):
        """Carrega um ensemble salvo."""
        with open(path, 'rb') as fh:
            return pickle.load(fh)

In [ ]:
# ---------------------------------------------------------------------------
# Construir o Champion EnsembleModel
# ---------------------------------------------------------------------------
if champion_name == 'simple_avg':
    champion_model = _EnsembleModel(base_models=models, mode='simple_avg')
elif champion_name == 'blend':
    champion_model = _EnsembleModel(base_models=models, mode='blend', weights=optimal_weights)
elif champion_name == 'stack':
    champion_model = _EnsembleModel(base_models=models, mode='stack', meta_learner=meta_learner)

# Métricas finais
champion_preds = champion_model.predict_proba(X_oot)[:, 1]
train_preds = champion_model.predict_proba(df_train[features])[:, 1]

final_ks   = compute_ks(y_oot, champion_preds)
final_auc  = roc_auc_score(y_oot, champion_preds)
final_gini = compute_gini(y_oot, champion_preds)
final_psi  = compute_psi(train_preds, champion_preds)

print("=== Métricas Finais do Champion ===")
print(f"  Estratégia: {champion_name}")
print(f"  KS  OOT:  {final_ks:.4f}")
print(f"  AUC OOT:  {final_auc:.4f}")
print(f"  Gini OOT: {final_gini:.2f}")
print(f"  PSI:       {final_psi:.4f}")

## Quality Gate QG-05

Verificações obrigatórias para aprovação do modelo ensemble:
- **KS > 0.20** — poder discriminante mínimo
- **AUC > 0.65** — capacidade preditiva mínima
- **Gini > 30** — concentração de risco adequada
- **PSI < 0.25** — estabilidade entre treino e OOT

In [ ]:
# ---------------------------------------------------------------------------
# Quality Gate QG-05
# ---------------------------------------------------------------------------
gates = {
    'KS > 0.20':   final_ks > 0.20,
    'AUC > 0.65':  final_auc > 0.65,
    'Gini > 30':   final_gini > 30,
    'PSI < 0.25':  final_psi < 0.25,
}

print("=== Quality Gate QG-05 ===")
all_pass = True
for check, passed in gates.items():
    status = 'PASS' if passed else 'FAIL'
    if not passed:
        all_pass = False
    print(f"  [{status}] {check}")

print(f"\n>>> Resultado geral: {'APROVADO' if all_pass else 'REPROVADO'}")
if not all_pass:
    print("AVISO: O modelo não atende a todos os critérios do quality gate.")

In [ ]:
# ---------------------------------------------------------------------------
# Visualizações
# ---------------------------------------------------------------------------
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1) Gráfico de barras — KS por estratégia
strategy_names = list(strategies.keys())
ks_values = [strategies[s]['ks'] for s in strategy_names]
colors = ['#2196F3', '#FF9800', '#4CAF50']
axes[0].bar(strategy_names, ks_values, color=colors, edgecolor='black')
axes[0].set_title('KS OOT por Estratégia')
axes[0].set_ylabel('KS')
axes[0].axhline(y=0.20, color='red', linestyle='--', label='Threshold 0.20')
axes[0].legend()

# 2) Gráfico de pizza — Distribuição dos pesos (Blend)
axes[1].pie(optimal_weights, labels=model_names, autopct='%1.1f%%', startangle=90)
axes[1].set_title('Distribuição dos Pesos (Blend)')

# 3) Overlay das distribuições de score
axes[2].hist(pred_avg_oot, bins=50, alpha=0.4, label='Média Simples', density=True)
axes[2].hist(pred_blend_oot, bins=50, alpha=0.4, label='Blend', density=True)
axes[2].hist(pred_stack_oot, bins=50, alpha=0.4, label='Stack', density=True)
axes[2].set_title('Distribuição de Scores OOT')
axes[2].set_xlabel('Probabilidade')
axes[2].set_ylabel('Densidade')
axes[2].legend()

plt.tight_layout()
os.makedirs(os.path.join(ARTIFACT_DIR, 'plots'), exist_ok=True)
plt.savefig(os.path.join(ARTIFACT_DIR, 'plots', 'ensemble_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Gráficos salvos em artifacts/plots/ensemble_comparison.png")

In [ ]:
# ---------------------------------------------------------------------------
# Salvar artefatos
# ---------------------------------------------------------------------------
# 1) Salvar ensemble .pkl
ensemble_path = os.path.join(ARTIFACT_DIR, 'models', 'champion_ensemble.pkl')
champion_model.save(ensemble_path)

# 2) Salvar resultados do ensemble
ensemble_results = {
    'champion_strategy': champion_name,
    'strategies': {},
    'quality_gate': {k: bool(v) for k, v in gates.items()},
    'quality_gate_passed': all_pass,
    'base_models': model_names,
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
}

for s_name, s_vals in strategies.items():
    ensemble_results['strategies'][s_name] = {
        'ks_oot': round(s_vals['ks'], 4),
        'auc_oot': round(s_vals['auc'], 4),
        'gini_oot': round(s_vals['gini'], 2),
    }

if champion_name == 'blend':
    ensemble_results['blend_weights'] = {n: round(w, 4) for n, w in zip(model_names, optimal_weights)}

results_path = os.path.join(ARTIFACT_DIR, 'ensemble_results.json')
with open(results_path, 'w') as fh:
    json.dump(ensemble_results, fh, indent=2)

# 3) Salvar metadata do champion
champion_metadata = {
    'model_type': 'ensemble',
    'strategy': champion_name,
    'n_base_models': len(model_names),
    'base_models': model_names,
    'metrics': {
        'ks_oot': round(final_ks, 4),
        'auc_oot': round(final_auc, 4),
        'gini_oot': round(final_gini, 2),
        'psi': round(final_psi, 4),
    },
    'n_features': len(features),
    'target': TARGET,
    'train_safras': TRAIN_SAFRAS,
    'oos_safra': OOS_SAFRA,
    'oot_safras': OOT_SAFRAS,
    'quality_gate_passed': all_pass,
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
}

metadata_path = os.path.join(ARTIFACT_DIR, 'champion_metadata.json')
with open(metadata_path, 'w') as fh:
    json.dump(champion_metadata, fh, indent=2)

print(f"\nArtefatos salvos:")
print(f"  Ensemble model:  {ensemble_path}")
print(f"  Resultados:      {results_path}")
print(f"  Metadata:        {metadata_path}")

log(f"07_ensemble | Champion: {champion_name} | KS={final_ks:.4f} AUC={final_auc:.4f} Gini={final_gini:.2f} PSI={final_psi:.4f} | QG={'PASS' if all_pass else 'FAIL'}")